<a href="https://colab.research.google.com/github/manikanta-eng/block-chain-/blob/main/assignment_9_block_chain_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# DEFI PROTOCOL SIMULATION USING ETHEREUM BLOCKCHAIN
# Blockchain Engineering Practical
# Deposit ETH, Withdraw ETH and Track User Balances
# Google Colab - Complete Single Cell Implementation
# ============================================================

import sys
import subprocess
import time

print("Installing required libraries...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "web3", "eth-tester[py-evm]", "py-solc-x"
])

from solcx import install_solc, compile_source
from web3 import Web3, EthereumTesterProvider
from web3.exceptions import ContractLogicError

print("\nLibraries installed successfully!")

# ------------------------------------------------------------
# STEP 1: CONNECT TO LOCAL ETHEREUM BLOCKCHAIN
# ------------------------------------------------------------

w3 = Web3(EthereumTesterProvider())

assert w3.is_connected(), "Blockchain connection failed!"

account = w3.eth.accounts[0]

print("\nSTEP 1: BLOCKCHAIN CONNECTION")
print("--------------------------------")
print("Blockchain Status:", "Connected")
print("Network: Local Ethereum Test Network")
print("Account:", account)
print("Initial Account Balance:",
      w3.from_wei(w3.eth.get_balance(account), "ether"), "ETH")

# ------------------------------------------------------------
# STEP 2: INSTALL SOLIDITY COMPILER
# ------------------------------------------------------------

print("\nInstalling Solidity Compiler 0.8.20...")
install_solc("0.8.20")

# ------------------------------------------------------------
# STEP 3: SOLIDITY SMART CONTRACT
# ------------------------------------------------------------

solidity_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

// Simple Mock DeFi Lending Contract
contract MockDeFiLending {

    // Store the ETH balance of every user
    mapping(address => uint256) public balances;

    // Track total ETH deposited
    uint256 public totalDeposits;

    // Event generated after deposit
    event Deposited(address indexed user, uint256 amount);

    // Event generated after withdrawal
    event Withdrawn(address indexed user, uint256 amount);

    // Deposit ETH into the lending contract
    function deposit() public payable {

        require(msg.value > 0, "Deposit must be greater than zero");

        balances[msg.sender] += msg.value;

        totalDeposits += msg.value;

        emit Deposited(msg.sender, msg.value);
    }

    // Withdraw ETH from the lending contract
    function withdraw(uint256 amount) public {

        require(amount > 0, "Amount must be greater than zero");

        require(
            balances[msg.sender] >= amount,
            "Insufficient user balance"
        );

        // Update balances before transferring ETH
        balances[msg.sender] -= amount;

        totalDeposits -= amount;

        // Transfer ETH to the user
        (bool success, ) = payable(msg.sender).call{value: amount}("");

        require(success, "ETH transfer failed");

        emit Withdrawn(msg.sender, amount);
    }

    // Check individual user balance
    function getBalance(address user)
        public
        view
        returns(uint256)
    {
        return balances[user];
    }

    // Check contract ETH balance
    function getContractBalance()
        public
        view
        returns(uint256)
    {
        return address(this).balance;
    }
}
"""

print("\nSTEP 2: SMART CONTRACT")
print("--------------------------------")
print("Solidity contract created successfully!")

# ------------------------------------------------------------
# STEP 4: COMPILE SMART CONTRACT
# ------------------------------------------------------------

compiled_sol = compile_source(
    solidity_code,
    output_values=["abi", "bin"],
    solc_version="0.8.20"
)

contract_interface = compiled_sol[
    "<stdin>:MockDeFiLending"
]

abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

print("Compilation Status: SUCCESS")
print("Contract ABI generated")
print("Contract Bytecode generated")

# ------------------------------------------------------------
# STEP 5: DEPLOY SMART CONTRACT
# ------------------------------------------------------------

DeFiContract = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

deploy_tx = DeFiContract.constructor().transact({
    "from": account
})

deploy_receipt = w3.eth.wait_for_transaction_receipt(
    deploy_tx
)

contract_address = deploy_receipt.contractAddress

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("\nSTEP 3: CONTRACT DEPLOYMENT")
print("--------------------------------")
print("Deployment Status: SUCCESS")
print("Contract Address:", contract_address)
print("Transaction Hash:", deploy_receipt.transactionHash.hex())

# ------------------------------------------------------------
# STEP 6: DEPOSIT 1 ETH
# ------------------------------------------------------------

deposit_amount = w3.to_wei(1, "ether")

deposit_tx = contract.functions.deposit().transact({
    "from": account,
    "value": deposit_amount
})

deposit_receipt = w3.eth.wait_for_transaction_receipt(
    deposit_tx
)

user_balance = contract.functions.getBalance(
    account
).call()

print("\nSTEP 4: DEPOSIT OPERATION")
print("--------------------------------")
print("Deposited Amount: 1 ETH")
print("Transaction Status:", deposit_receipt.status)
print("User Balance:",
      w3.from_wei(user_balance, "ether"), "ETH")

assert deposit_receipt.status == 1
assert user_balance == deposit_amount

# ------------------------------------------------------------
# STEP 7: WITHDRAW 0.4 ETH
# ------------------------------------------------------------

withdraw_amount = w3.to_wei(0.4, "ether")

withdraw_tx = contract.functions.withdraw(
    withdraw_amount
).transact({
    "from": account
})

withdraw_receipt = w3.eth.wait_for_transaction_receipt(
    withdraw_tx
)

remaining_balance = contract.functions.getBalance(
    account
).call()

print("\nSTEP 5: WITHDRAWAL OPERATION")
print("--------------------------------")
print("Withdrawn Amount: 0.4 ETH")
print("Transaction Status:", withdraw_receipt.status)
print("Remaining User Balance:",
      w3.from_wei(remaining_balance, "ether"), "ETH")

assert withdraw_receipt.status == 1
assert remaining_balance == w3.to_wei(0.6, "ether")

# ------------------------------------------------------------
# STEP 8: CHECK CONTRACT BALANCE
# ------------------------------------------------------------

contract_balance = contract.functions.getContractBalance().call()

total_deposits = contract.functions.totalDeposits().call()

print("\nSTEP 6: CONTRACT BALANCE")
print("--------------------------------")
print("Contract ETH Balance:",
      w3.from_wei(contract_balance, "ether"), "ETH")

print("Total Deposits:",
      w3.from_wei(total_deposits, "ether"), "ETH")

assert contract_balance == remaining_balance
assert total_deposits == remaining_balance

# ------------------------------------------------------------
# STEP 9: TEST INSUFFICIENT WITHDRAWAL
# ------------------------------------------------------------

print("\nSTEP 7: NEGATIVE TESTING")
print("--------------------------------")

try:
    contract.functions.withdraw(
        w3.to_wei(2, "ether")
    ).call({"from": account})

    print("Unexpected result: Withdrawal was allowed!")

except Exception:
    print("Insufficient balance test: PASSED")
    print("Correctly rejected withdrawal exceeding user balance.")

# ------------------------------------------------------------
# STEP 10: FINAL RESULTS
# ------------------------------------------------------------

print("\n============================================")
print("          FINAL PRACTICAL RESULTS")
print("============================================")

print("Blockchain Connection       : SUCCESS")
print("Solidity Compilation        : SUCCESS")
print("Smart Contract Deployment   : SUCCESS")
print("ETH Deposit                 : SUCCESS")
print("ETH Withdrawal              : SUCCESS")
print("User Balance Tracking       : SUCCESS")
print("Contract Balance Tracking   : SUCCESS")
print("Insufficient Balance Test   : SUCCESS")

print("--------------------------------------------")
print("Initial Deposit       : 1.0 ETH")
print("Withdrawal            : 0.4 ETH")
print("Remaining Balance     : 0.6 ETH")
print("Contract Balance      : 0.6 ETH")
print("--------------------------------------------")

print("ALL PRACTICAL TESTS COMPLETED SUCCESSFULLY!")
print("============================================")


Installing required libraries...

Libraries installed successfully!

STEP 1: BLOCKCHAIN CONNECTION
--------------------------------
Blockchain Status: Connected
Network: Local Ethereum Test Network
Account: 0x7E5F4552091A69125d5DfCb7b8C2659029395Bdf
Initial Account Balance: 1000000 ETH

Installing Solidity Compiler 0.8.20...

STEP 2: SMART CONTRACT
--------------------------------
Solidity contract created successfully!
Compilation Status: SUCCESS
Contract ABI generated
Contract Bytecode generated


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



STEP 3: CONTRACT DEPLOYMENT
--------------------------------
Deployment Status: SUCCESS
Contract Address: 0xF2E246BB76DF876Cef8b38ae84130F4F55De395b
Transaction Hash: 2bb2f2a10c7843bcc5ea591e6e67628419a5a6cfd3c584932b6edc1254d41d45

STEP 4: DEPOSIT OPERATION
--------------------------------
Deposited Amount: 1 ETH
Transaction Status: 1
User Balance: 1 ETH

STEP 5: WITHDRAWAL OPERATION
--------------------------------
Withdrawn Amount: 0.4 ETH
Transaction Status: 1
Remaining User Balance: 0.6 ETH

STEP 6: CONTRACT BALANCE
--------------------------------
Contract ETH Balance: 0.6 ETH
Total Deposits: 0.6 ETH

STEP 7: NEGATIVE TESTING
--------------------------------
Insufficient balance test: PASSED
Correctly rejected withdrawal exceeding user balance.

          FINAL PRACTICAL RESULTS
Blockchain Connection       : SUCCESS
Solidity Compilation        : SUCCESS
Smart Contract Deployment   : SUCCESS
ETH Deposit                 : SUCCESS
ETH Withdrawal              : SUCCESS
User Balance 